# Ambiguous-intent stress comparison

Upload phase4_ambiguous_source.zip to My Drive root. Choose A100 and run all.
Uses uv, a separate ZIP, and the actual production search path at
hybrid / rerank20 / rerank50.

These 15 questions omit test/symbol cues but target previously inspected behaviours.
Alternatives accept implementation or a complete demonstration. This is a post-hoc
stress set, not an independent holdout. No tuning or default promotion.
Deployment hardware remains undecided; A100 timing is experimental only.

In [ ]:
SOURCE_ZIP = "/content/drive/MyDrive/phase4_ambiguous_source.zip"

In [ ]:
# Load the separate source ZIP that you uploaded to Google Drive.
import hashlib, pathlib, zipfile
from google.colab import drive

drive.mount("/content/drive")
source_zip = pathlib.Path(SOURCE_ZIP)
if not source_zip.is_file():
    raise FileNotFoundError(
        f"Upload phase4_ambiguous_source.zip to My Drive, or update SOURCE_ZIP. Missing: {source_zip}"
    )
expected_sha256 = "51eaef7af328bc500d7015aadc00ba7f565e517ce4a802e4e97e9e764e23a01c"
if hashlib.sha256(source_zip.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError(
        "Source ZIP does not match this notebook. Download the matching ZIP and notebook again."
    )
PAYLOAD = pathlib.Path("/content/phase4_ambiguous_payload")
with zipfile.ZipFile(source_zip) as package:
    package.extractall(PAYLOAD)
print("Loaded project code and question datasets from", source_zip.name)

In [ ]:
import importlib.util, subprocess, sys
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)
if importlib.util.find_spec('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--system',
                '--python', sys.executable, '-q', 'chromadb==1.5.9', 'fastapi==0.162.0', 'gitpython==3.2.1', 'langchain-chroma==1.1.0', 'langchain-core==1.6.13', 'langchain-text-splitters==1.1.3', 'langgraph==1.2.19', 'pydantic==2.14.1', 'pydantic-settings==2.16.1', 'rank-bm25==0.2.2', 'rich==15.0.0', 'sentence-transformers==5.7.0', 'transformers==4.57.6', 'tree-sitter==0.26.0', 'tree-sitter-language-pack==1.22.1', 'typer==0.27.3', 'uvicorn==0.54.0', 'astchunk==0.1.0'], check=True)

In [ ]:
import json, os, pathlib, subprocess, sys
REPOS = pathlib.Path('/content/ambiguous_repos')
RESULTS = pathlib.Path('/content/ambiguous_results')
REPOS.mkdir(exist_ok=True)
env = {**os.environ, 'PYTHONPATH': os.pathsep.join([str(PAYLOAD), str(PAYLOAD / 'src')]),
       'TOKENIZERS_PARALLELISM': 'false'}
manifest = json.loads((PAYLOAD / 'eval/datasets/phase4_ambiguous/manifest.json').read_text())
for entry in manifest['datasets']:
    repo = entry['repo']
    checkout = REPOS / repo.replace('/', '__')
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout',
                        'https://github.com/' + repo, str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach',
                    entry['commit_sha']], check=True)
    dataset = PAYLOAD / 'eval/datasets/phase4_ambiguous' / entry['file']
    if hashlib.sha256(dataset.read_bytes()).hexdigest() != entry['sha256']:
        raise ValueError('Dataset checksum mismatch')
    subprocess.run([sys.executable, str(PAYLOAD / 'eval/run_ambiguous_eval.py'),
                    '--dataset', str(dataset), '--checkout', str(checkout),
                    '--output', str(RESULTS / repo.replace('/', '__')),
                    '--mode', 'full', '--device', 'cuda', '--repetitions', '3'],
                    env=env, check=True)

In [ ]:
from google.colab import files
archive = pathlib.Path('/content/phase4_ambiguous_results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULTS.rglob('*')):
        if path.is_file() and 'isolated_index' not in path.parts:
            bundle.write(path, path.relative_to(RESULTS))
files.download(str(archive))